In [1]:
import os
import json
from glob import glob
from pathlib import Path
from urllib.parse import urlparse

import httpx
from tqdm import tqdm
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset, load_dataset


ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"

/home/octoopt/workspace/projects/personal/data_enrichment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [47]:
def download_image(image_url, save_path, timeout=10):
    """
    Download an image from a URL and save it to the specified path using httpx.

    Args:
        image_url (str): URL of the image to download
        save_path (str): Local path where to save the image (including filename)
        timeout (int): Request timeout in seconds

    Returns:
        bool: True if download successful, False otherwise
    """
    try:
        with httpx.Client() as client:
            response = client.get(image_url, timeout=timeout)
            response.raise_for_status()  # Raise an exception for bad status codes

            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            # Write the image to file
            with open(save_path, "wb") as file:
                file.write(response.content)

        # print(f"✓ Downloaded: {save_path}")
        return True

    except httpx.RequestError as e:
        print(f"✗ Failed to download {image_url}: {e}")
        return False
    except Exception as e:
        print(f"✗ Error saving image: {e}")
        return False


def read_json(file_path):
    """Read and return data from a JSON file."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in {file_path}: {e}")
        return None


def write_json(data, file_path, indent=4):
    """Write data to a JSON file."""
    try:
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)
        print(f"✓ Written JSON to: {file_path}")
        return True
    except Exception as e:
        print(f"✗ Error writing JSON: {e}")
        return False

In [5]:
# datafile = str(DATA_DIR / "german" / "politicians")
datafile = str(DATA_DIR / "vietnam")
datapaths = glob(datafile + "/**.json")
datapaths

['/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinuni_cas_faculty_sections.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/ussh_teachers.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/fpt_teachers.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/danhbaluatsu.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinuni_cas_faculty.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinuni_people.json',
 '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinmec_doctors.json']

### Profiling

Normalize to this format

```
from dataclasses import dataclass

@dataclass
class Profile:
    name: str
    mainType: str # Job
    dateOfBirth: str
    homePlace: str  # where the person born from
    workPlace: str
    gender: str
    image_url: str
    profile_url: str
    other_info: dict

```

In [18]:
# ussh_profiles
norm_ussh_profiles = []
ussh_profiles = read_json(file_path=datapaths[1])


for data in ussh_profiles:
    profile = {}
    profile['name'] = data.get('name', '')
    profile['mainType'] = data.get('title', '')
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] =data.get('homePlace', '')
    profile['workPlace'] = "Trường Đại học Khoa học Xã hội và Nhân văn"
    profile['gender'] =data.get('gender', '')
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')

    norm_ussh_profiles.append(profile)

norm_ussh_profiles[0]

{'name': 'ThS. Nguyễn Thị Huệ',
 'mainType': 'Giám đốc Trung tâm',
 'dateOfBirth': '',
 'homePlace': '',
 'workPlace': 'Trường Đại học Khoa học Xã hội và Nhân văn',
 'gender': '',
 'image_url': 'https://daotaonhanluc.hcmussh.edu.vn/wp-content/uploads/2023/04/admin-ajax.jpg',
 'profile_url': 'https://daotaonhanluc.hcmussh.edu.vn/giang-vien/nguyen-thi-hue/'}

In [19]:
# fpt_profiles

norm_fpt_profiles = []
fpt_profiles = read_json(file_path=datapaths[2])


for data in fpt_profiles:
    profile = {}
    profile['name'] = data.get('name', '')
    profile['mainType'] = data.get('role', '')
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] =data.get('homePlace', '')
    profile['workPlace'] = "FPT Schools - 15 Đông Quan, phường Nghĩa Đô, HN"
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')


    if "Cô" in profile['name']: 
        gender = "Female"
    elif "Thầy" in profile['name']: 
        gender = "Male"
    else:
        gender = ""

    profile['gender'] = gender
    norm_fpt_profiles.append(profile)


# fpt_profiles
norm_fpt_profiles[0]

{'name': '่Cô Đinh Vũ Hoài Nga',
 'mainType': 'Tổ trưởng tổ PDP',
 'dateOfBirth': '',
 'homePlace': '',
 'workPlace': 'FPT Schools - 15 Đông Quan, phường Nghĩa Đô, HN',
 'image_url': "data:image/svg+xml,%3Csvg%20xmlns='http://www.w3.org/2000/svg'%20viewBox='0%200%20299%20450'%3E%3C/svg%3E",
 'profile_url': 'https://hanoi-school.fpt.edu.vn/giao-vien/%e0%b9%88co-dinh-vu-hoai-nga',
 'gender': 'Female'}

In [33]:
# danhbaluatsu_profiles

# fpt_profiles

norm_danhbaluatsu_profiles = []
danhbaluatsu_profiles = read_json(file_path=datapaths[3])

def normalize(data: str): 
    data = data.replace(':  ', ':')
    return data

for data in danhbaluatsu_profiles:
    profile = {}
    profile['name'] = data.get('name', '')

    # work profile
    the_luat_su = normalize(data['Thẻ luật sư']).replace(':', '')
    chung_chi = normalize(data['Chứng chỉ ngành nghề số']).replace(':', '')
    doan_luat_su = normalize(data['Đoàn luật sư']).replace(': ', '')
    to_chuc = normalize(data['Tổ chức hành nghề']).replace(': ', '')


    profile['mainType'] = "Luật sư"
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] =data.get('homePlace', '')
    profile['workplace'] = f"{doan_luat_su} - {to_chuc}"
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')

    other_info = {
        "the_luat_su": the_luat_su, 
        "chung_chi": chung_chi, 
        "doan_luat_su": doan_luat_su, 
        "to_chuc": to_chuc
    }

    profile['other_info'] = other_info



    profile['gender'] = profile.get('gender', '')
    norm_danhbaluatsu_profiles.append(profile)


# danhbaluatsu_profiles
norm_danhbaluatsu_profiles[0]

{'name': 'Nguyễn Ngọc Thành',
 'mainType': 'Luật sư',
 'dateOfBirth': '',
 'homePlace': '',
 'workplace': 'TP. Hồ Chí Minh - Liên đoàn luật sư Việt nam',
 'image_url': 'https://www.danhbaluatsu.com/images/news/',
 'profile_url': 'https://www.danhbaluatsu.com/luat-su/nguyen-ngoc-thanh/',
 'other_info': {'the_luat_su': '9095/LS',
  'chung_chi': '10.577/TP/LS-CCHN',
  'doan_luat_su': 'TP. Hồ Chí Minh',
  'to_chuc': 'Liên đoàn luật sư Việt nam'},
 'gender': ''}

In [36]:
# vinmec_profile

norm_vinmec_profiles = []
vinmec_profile = read_json(file_path="/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinmec_doctors.json")

# def normalize(data: str): 
#     data = data.replace(':  ', ':')
#     return data

for data in vinmec_profile:
    profile = {}
    profile['name'] = data.get('name', '')

    # work profile
    degree = data.get('degree')
    specialty = data['specialty']


    profile['mainType'] = f"{degree} - {specialty}"
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] = data.get('homePlace', '')
    profile['workplace'] = data.get('hospital', '')
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')

    profile['gender'] = profile.get('gender', '')
    norm_vinmec_profiles.append(profile)


# vinmec_profile
norm_vinmec_profiles[0]

{'name': 'Cố vấn chuyên môn Phó Hoàng Đăng Mịch',
 'mainType': 'Giáo sư,  Phó giáo sư, Tiến sĩ, - Nội tiết',
 'dateOfBirth': '',
 'homePlace': '',
 'workplace': 'Khoa khám bệnh & Nội khoa - Bệnh viện Đa khoa Vinmec Hạ Long',
 'image_url': 'https://www.vinmec.com/static/uploads/small_dr_hoang_dang_mich_039eb0360b.JPG',
 'profile_url': 'https://www.vinmec.com/vie/chuyen-gia-y-te/hoang-dang-mich-43322-vi',
 'gender': ''}

In [41]:
# vinuni_profile

norm_vinuni_profiles = []
vinuni_profile = read_json(file_path="/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinuni_people.json")

# def normalize(data: str): 
#     data = data.replace(':  ', ':')
#     return data

for data in vinuni_profile:
    profile = {}
    profile['name'] = data.get('name', '')

    profile['mainType'] = data.get('position', '')
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] = data.get('homePlace', '')
    profile['workplace'] = data.get('workplace', '')
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')

    profile['gender'] = profile.get('gender', '')
    norm_vinuni_profiles.append(profile)


# vinuni_profile
norm_vinuni_profiles[0]

{'name': 'Nguyễn Ngọc Minh',
 'mainType': 'Phó Tổng Giám đốc Khối Học thuật, Hệ thống Giáo dục Vinschool',
 'dateOfBirth': '',
 'homePlace': '',
 'workplace': '',
 'image_url': 'https://vinuni.edu.vn/wp-content/uploads/2025/10/Profile-pic-3.png',
 'profile_url': '',
 'gender': ''}

In [40]:
# vinuni_cas_faculty

norm_vinuni_cas_faculty_profiles = []
vinuni_cas_faculty_profile = read_json(file_path= '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinuni_cas_faculty.json')


for data in vinuni_cas_faculty_profile:
    profile = {}
    profile['name'] = data.get('name', '')

    profile['mainType'] = data.get('position', '')
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] = data.get('homePlace', '')
    profile['workplace'] = data.get('workplace', '')
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')

    profile['gender'] = profile.get('gender', '')
    norm_vinuni_cas_faculty_profiles.append(profile)


# vinuni_cas_faculty_profile
norm_vinuni_cas_faculty_profiles[0]

{'name': 'GS. Umberto Ansaldo',
 'mainType': 'Viện trưởng',
 'dateOfBirth': '',
 'homePlace': '',
 'workplace': 'Viện Khoa học và Giáo dục Khai phóng (College of Arts and Sciences – CAS)',
 'image_url': 'https://cas.vinuni.edu.vn/wp-content/uploads/2025/02/Thiet-ke-chua-co-ten-18-e1744168228564.png',
 'profile_url': 'https://vinuni.edu.vn/vi/people/umberto-ansaldo/',
 'gender': ''}

In [43]:
# vinuni_cas_faculty

norm_vinuni_cas_sect_profiles = []
vinuni_cas_sect_profile = read_json(file_path= '/home/octoopt/workspace/projects/personal/data_enrichment/data/vietnam/vinuni_cas_faculty_sections.json',)


for data in vinuni_cas_sect_profile:
    profile = {}
    profile['name'] = data.get('name', '')

    profile['mainType'] = data.get('position', '')
    profile['dateOfBirth'] =data.get('dateOfBirth', '')
    profile['homePlace'] = data.get('homePlace', '')
    profile['workplace'] = data.get('workplace', '')
    profile['image_url'] =data.get('image_url', '')
    profile['profile_url'] =data.get('profile_url', '')

    profile['gender'] = profile.get('gender', '')
    norm_vinuni_cas_sect_profiles.append(profile)


# vinuni_cas_sect_profile
norm_vinuni_cas_sect_profiles[0]

{'name': 'GS. Umberto Ansaldo',
 'mainType': 'Viện trưởng',
 'dateOfBirth': '',
 'homePlace': '',
 'workplace': 'Viện Khoa học và Giáo dục Khai phóng (College of Arts and Sciences – CAS)',
 'image_url': 'https://cas.vinuni.edu.vn/wp-content/uploads/2025/02/Thiet-ke-chua-co-ten-18-e1744168228564.png',
 'profile_url': 'https://vinuni.edu.vn/vi/people/umberto-ansaldo/',
 'gender': ''}

In [44]:
end_source_profiles = [
    norm_danhbaluatsu_profiles, 
    norm_fpt_profiles, 
    norm_ussh_profiles, 
    norm_vinmec_profiles, 
    norm_vinuni_cas_faculty_profiles, 
    norm_vinuni_cas_sect_profiles, 
    norm_vinuni_profiles
]

end_profiles = []

for profiles in end_source_profiles:
    end_profiles += profiles


len(end_profiles)

4307

In [46]:
write_json(
    data=end_profiles, 
        file_path='../data/vietnamese_profiles_05102025_before.json', 
)

✓ Written JSON to: ../data/vietnamese_profiles_05102025_before.json


True

In [ ]:
"""
Only get profile that get image
"""

usable_profiles = []
save_dir = str(DATA_DIR / "images" / "vietnamese_profiles")


for idx in tqdm(range(len(end_profiles))):
    try:
        profile = end_profiles[idx]
        image_url = profile['image_url']
        img_name = profile["name"].replace(" ", "_")
        local_path = f"{save_dir}/{img_name}.jpg"
        profile['local_path'] = local_path
        is_success = download_image(image_url, local_path)
        if is_success:
            usable_profiles.append(profile)

    except Exception as e:
        print(e)
        continue

In [51]:
write_json(
    data=usable_profiles, 
        file_path='../data/vietnamese_profiles_05102025_after.json', 
)

✓ Written JSON to: ../data/vietnamese_profiles_05102025_after.json


True